# Tool Usage Basic

In [1]:
import sys
print(sys.executable)

d:\Code\swe_agent_jom\.venv\Scripts\python.exe


In [2]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", deepseek_api_key is not None)

DEEPSEEK_API_KEY loaded: True


In [3]:
import os
import json
from typing import cast

from openai import OpenAI
from openai.types.chat import (
    ChatCompletionMessageParam,
    ChatCompletionAssistantMessageParam,
    ChatCompletionToolMessageParam,
)
from openai.types.chat.chat_completion_tool_union_param import (
    ChatCompletionToolUnionParam,
)
from openai.types.chat.chat_completion_message_function_tool_call import (
    ChatCompletionMessageFunctionToolCall,
)


client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)


birthday_table = {
    "Jack": "3月18日",
    "Alice": "7月2日",
    "Bob": "11月9日",
    "Tom": "1月25日",
}


def get_birthday(name: str) -> str:
    return birthday_table.get(name, "没有查到这个人的生日")


tools: list[ChatCompletionToolUnionParam] = [
    {
        "type": "function",
        "function": {
            "name": "get_birthday",
            "description": "查询本地生日表，返回某个人的生日",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {
                        "type": "string",
                        "description": "人名，例如 Jack、Alice、Bob、Tom",
                    }
                },
                "required": ["name"],
            },
        },
    }
]


messages: list[ChatCompletionMessageParam] = [
    {
        "role": "system",
        "content": "你是一个生日查询助手。用户问生日时，必须调用 get_birthday 工具。",
    },
    {
        "role": "user",
        "content": "Jack 的生日是几月几号？",
    },
]


response = client.chat.completions.create(
    model="deepseek-v4-flash",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)


assistant_message = response.choices[0].message

assistant_param = cast(
    ChatCompletionAssistantMessageParam,
    assistant_message.model_dump(exclude_none=True),
)

messages.append(assistant_param)


if assistant_message.tool_calls:
    for raw_tool_call in assistant_message.tool_calls:
        if raw_tool_call.type != "function":
            continue

        tool_call = cast(ChatCompletionMessageFunctionToolCall, raw_tool_call)

        tool_name = tool_call.function.name
        tool_args = json.loads(tool_call.function.arguments)

        if tool_name == "get_birthday":
            result = get_birthday(tool_args["name"])

            tool_message: ChatCompletionToolMessageParam = {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result,
            }

            messages.append(tool_message)

    final_response = client.chat.completions.create(
        model="deepseek-v4-flash",
        messages=messages,
    )

    print(final_response.choices[0].message.content)

else:
    print(assistant_message.content)

Jack的生日是 **3月18日**。🎂
